# Entraînement du modèle de classification des tumeurs cérébrales
Ce notebook utilise le module `src/train.py` pour entraîner un CNN sur les données TFRecord préparées par Spark.

In [12]:
import sys
from pathlib import Path
import json

# Ajouter le dossier parent au path pour importer les modules du projet
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Imports du projet
from train import load_settings, run_training

import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"Project root: {project_root}")

TensorFlow version: 2.20.0
GPU disponible: False
Project root: c:\Code\Cours\sparkcore\Brain-Tumor-MRI-Classification


## 1. Configuration

Le fichier `conf/train.yaml` contient tous les hyperparamètres. Vous pouvez le modifier ou créer une config personnalisée ici.

In [13]:
# Charger la configuration du projet
config_path = "../conf/train.yaml"
settings = load_settings(config_path)

# Afficher la configuration
print("Configuration chargée:")
for key, value in settings.items():
    print(f"  {key}: {value}")

Configuration chargée:
  input_tfrecord_path: data/processed/training_tfrecord/current
  seed: 42
  image_height: 224
  image_width: 224
  batch_size: 32
  epochs: 30
  learning_rate: 0.001
  shuffle_buffer: 2048
  num_classes: None
  steps_per_epoch: None
  validation_steps: None
  early_stopping_patience: 10
  model_output_dir: models\baseline_cnn
  best_model_path: models\baseline_cnn\best.keras
  final_model_path: models\baseline_cnn\final.keras
  run_summary_path: models\baseline_cnn\train_run_summary.json


## 2. Modification de la configuration (optionnel)

Pour un test rapide, on peut réduire les époques et le batch size.

In [14]:
# Ajuster pour un test rapide (décommenter si besoin)
settings["epochs"] = 5  # Réduire pour test rapide
settings["batch_size"] = 16
# settings["steps_per_epoch"] = 50  # Limiter les steps pour tester rapidement
# settings["validation_steps"] = 20

print("\nConfiguration modifiée:")
print(f"  epochs: {settings['epochs']}")
print(f"  batch_size: {settings['batch_size']}")
print(f"  learning_rate: {settings['learning_rate']}")
print(f"  image_size: {settings['image_height']}x{settings['image_width']}")


Configuration modifiée:
  epochs: 5
  batch_size: 16
  learning_rate: 0.001
  image_size: 224x224


## 3. Vérifier les fichiers TFRecord disponibles

In [ ]:
# Vérifier que les fichiers existent
tfrecord_path = Path(settings["input_tfrecord_path"])
train_files = sorted(tfrecord_path.glob("split=train/shard_id=*/*.tfrecord"))
val_files = sorted(tfrecord_path.glob("split=val/shard_id=*/*.tfrecord"))
test_files = sorted(tfrecord_path.glob("split=test/shard_id=*/*.tfrecord"))

print(f"\n📁 TFRecords dans {tfrecord_path}:")
print(f"  Train: {len(train_files)} fichiers")
print(f"  Val: {len(val_files)} fichiers")
print(f"  Test: {len(test_files)} fichiers")

if not train_files:
    print(f"❌ Aucun fichier TFRecord trouvé. Lancez d'abord les jobs Spark de préparation.")
else:
    print("✅ Fichiers TFRecord prêts")

## 4. Lancer l'entraînement

La fonction `run_training()` du module `train.py` gère tout le pipeline:
- Chargement des TFRecords
- Construction du modèle CNN
- Entraînement avec callbacks (ModelCheckpoint, EarlyStopping)
- Évaluation sur le test set
- Sauvegarde des modèles et du résumé

In [16]:
# Lancer l'entraînement
print("🚀 Démarrage de l'entraînement...\n")
print("="*60)

result = run_training(settings)

print("="*60)
print("\n✅ Entraînement terminé !")

🚀 Démarrage de l'entraînement...



FileNotFoundError: TFRecord root path not found: C:\Code\Cours\sparkcore\Brain-Tumor-MRI-Classification\notebooks\data\processed\training_tfrecord\current

## 5. Résultats de l'entraînement

In [ ]:
# Afficher les statistiques du dataset
print("\n📊 Statistiques du dataset:")
print(f"  Total d'images: {result['rows_total']}")
print(f"  Train: {result['rows_train']} images")
print(f"  Validation: {result['rows_val']} images")
print(f"  Test: {result['rows_test']} images")
print(f"  Nombre de classes: {result['num_classes_used']}")
print(f"\n  Fichiers TFRecord:")
print(f"    Train: {result['train_files_count']}")
print(f"    Val: {result['val_files_count']}")
print(f"    Test: {result['test_files_count']}")

## 6. Historique d'entraînement

In [ ]:
# Créer un DataFrame avec l'historique
history_df = pd.DataFrame(result['history'])
print(f"\n📈 Historique de l'entraînement ({len(history_df)} époques):\n")
print(history_df.to_string(index=True))

## 7. Visualisation des courbes d'entraînement

In [ ]:
# Tracer les courbes de loss et accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
epochs = range(1, len(history_df) + 1)
ax1.plot(epochs, history_df['loss'], label='Train Loss', marker='o')
if 'val_loss' in history_df.columns:
    ax1.plot(epochs, history_df['val_loss'], label='Val Loss', marker='s')
ax1.set_title('Loss pendant l\'entraînement', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs, history_df['accuracy'], label='Train Accuracy', marker='o')
if 'val_accuracy' in history_df.columns:
    ax2.plot(epochs, history_df['val_accuracy'], label='Val Accuracy', marker='s')
ax2.set_title('Accuracy pendant l\'entraînement', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()

# Sauvegarder dans le dossier du modèle
curves_path = Path(result['settings']['model_output_dir']) / 'training_curves.png'
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n💾 Graphiques sauvegardés: {curves_path}")

## 8. Métriques sur le test set

In [ ]:
# Afficher les métriques de test
if result['test_metrics']:
    print("\n📊 Résultats sur le test set:")
    for metric_name, metric_value in result['test_metrics'].items():
        print(f"  {metric_name}: {metric_value:.4f}", end="")
        if 'accuracy' in metric_name:
            print(f" ({metric_value*100:.2f}%)")
        else:
            print()
else:
    print("\n⚠️ Aucune métrique de test disponible")

## 9. Fichiers générés

In [ ]:
# Afficher les chemins des fichiers créés
print("\n💾 Fichiers générés:")
print(f"  Meilleur modèle: {result['best_model_path']}")
print(f"  Modèle final: {result['final_model_path']}")
print(f"  Résumé JSON: {result['settings']['run_summary_path']}")

# Vérifier que les fichiers existent
best_model = Path(result['best_model_path'])
final_model = Path(result['final_model_path'])
summary_file = Path(result['settings']['run_summary_path'])

if best_model.exists():
    print(f"\n✅ Meilleur modèle sauvegardé ({best_model.stat().st_size / 1024 / 1024:.2f} MB)")
if final_model.exists():
    print(f"✅ Modèle final sauvegardé ({final_model.stat().st_size / 1024 / 1024:.2f} MB)")
if summary_file.exists():
    print(f"✅ Résumé d'entraînement sauvegardé")

## 10. Charger le modèle et faire des prédictions

In [ ]:
# Charger le meilleur modèle
best_model_path = result['best_model_path']
model = tf.keras.models.load_model(best_model_path)

print(f"✅ Modèle chargé depuis: {best_model_path}")
print(f"\nArchitecture du modèle:")
model.summary()

## 🎉 Entraînement terminé !

Le notebook a utilisé le module `src/train.py` du projet pour entraîner le modèle.

**Prochaines étapes possibles:**
- Modifier `conf/train.yaml` pour ajuster les hyperparamètres
- Augmenter le nombre d'époques pour améliorer les performances
- Utiliser le modèle sauvegardé pour faire des prédictions
- Essayer des architectures plus complexes en modifiant `_build_model()` dans `train.py`